## CardiaSense AI — LLM integration

This notebook covers: **personalized narrative, recommendations, optional live LLM**.

Run `./setup.sh` first (creates the venv, installs deps, downloads the dataset if needed).
The dataset root is resolved from `CARDIASENSE_DATA_DIR` or `.env`.

> Educational / research prototype — not a medical device.


In [1]:

import os, sys
from pathlib import Path

ROOT = Path("/home/tnzr/CardiaSense-AI")
sys.path.insert(0, str(ROOT / "ml"))
sys.path.insert(0, str(ROOT))

DATA = os.environ.get("CARDIASENSE_DATA_DIR") or     "/media/tnzr/AuxVolume/datasets/HLS-CMDS Heart and Lung Sounds Dataset " \
    "Recorded from a Clinical Manikin using Digital Stethoscope"


### Personalized AI narrative (LLM, with offline fallback)

In [2]:

from webapp.app import llm as llm_mod
findings = [
    {"key":"heart_binary","label":"Heart screening","kind":"binary","pred":"abnormal","confidence":0.82,
     "classes":["normal","abnormal"],"probs":[0.18,0.82]},
    {"key":"lung_types","label":"Lung typing","kind":"multiclass","pred":"Wheezing","confidence":0.71,
     "classes":["Normal","Wheezing","Fine Crackles","Coarse Crackles","Rhonchi","Pleural Rub"]},
]
out = llm_mod.generate_narrative(findings, patient_context="58y, asthma, mild palpitations")
import json
print("provider:", out["provider"], "| model:", out["model"], "| personalized:", out["personalized"])
print(out["summary"])
for area, recs in out["recommendations"].items():
    print(f"  {area}: {recs}")


provider: synthesizer | model: rule-based-synthesizer-v1 | personalized: True
The model flagged at least one possible abnormality. A clinician should interpret these findings with your full history and an examination.
  medical: ['Personalized from your stated context: Patient context mentions asthma — airway findings should be weighed accordingly.; Patient reports palpitations — correlate with rhythm findings and ECG.', 'Evaluate for asthma/COPD with a clinician; use reliever inhalers as prescribed']
  diet: ['Avoid known food triggers if asthma is food-sensitive', 'Stay hydrated to keep airways moist']
  environment: ['Reduce smoke, dust and strong fumes', 'Keep indoor humidity moderate']


### Optional: live LLM via an OpenAI-compatible endpoint

In [3]:

if os.environ.get("CARDIASENSE_LLM_API_KEY"):
    narr = llm_mod.generate_narrative(findings, "asthma", [{"role":"user","content":"should I worry about the wheeze?"}])
    print("provider:", narr["provider"], narr.get("summary", "")[:200])
else:
    print("No CARDIASENSE_LLM_API_KEY — using the offline synthesizer (already shown above).")


No CARDIASENSE_LLM_API_KEY — using the offline synthesizer (already shown above).
